# Preconditions
`./setup_auth.ipynb` and `./setup_catalog_policies.ipynb`will run

In [101]:
%run ./setup_catalog_policies.ipynb

/Users/apabook/Desktop/ds-protocol/static/tutorial/venv/bin/python
{
    "participant_id": "did:jwk:provider",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1200",
    "token": null,
    "saved_at": "2026-03-24T08:23:27.578761",
    "last_interaction": "2026-03-24T08:23:27.578823",
    "is_me": true
}
{
    "participant_id": "did:jwk:consumer",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1100",
    "token": null,
    "saved_at": "2026-03-24T08:23:27.551672",
    "last_interaction": "2026-03-24T08:23:27.551707",
    "is_me": true
}
Provider DID: did:jwk:provider

Provider token: token

Consumer DID: did:jwk:consumer

Consumer token: token
{
    "dctConformsTo": null,
    "dctCreator": null,
    "dctIdentifier": "urn:catalog:0689bcec-514f-48ca-b168-14ae10a84364",
    "dctIssued": "2026-03-24T08:23:27.656914Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainCatalo

# Contract Negotiation

## Initialization of negotiation request (Consumer -> Provider)

In [102]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:9e4ed662-3d11-428b-a229-e396620b8102",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8b0488c4-d4ba-4464-9fe7-83ae68bcc2ca"
    }
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e",
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:f182

## Provider creates initial offer (Provider -> Consumer)

In [103]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:9e4ed662-3d11-428b-a229-e396620b8102",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8b0488c4-d4ba-4464-9fe7-83ae68bcc2ca"
    },
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e",
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:f1141641-3d14-471a-a4

## Consumer sends negotiation request based on offer (Consumer -> Provider)

In [104]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:9e4ed662-3d11-428b-a229-e396620b8102",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8b0488c4-d4ba-4464-9fe7-83ae68bcc2ca"
    },
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e",
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:f1829f9c-cf14-4786-a2c9-9275bb5d83a1",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http

## Provider updates/confirms the offer (Provider -> Consumer)

In [105]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:9e4ed662-3d11-428b-a229-e396620b8102",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8b0488c4-d4ba-4464-9fe7-83ae68bcc2ca"
    },
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e",
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:f1141641-3d14-471a-a409-515a3a61860d",
    "state": "OFFERED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": 

## Consumer accepts the offer (Consumer -> Provider)

In [106]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e",
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:f1829f9c-cf14-4786-a2c9-9275bb5d83a1",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-25T10:53:20.546655Z",
    "updatedAt": "2026-03-25T10:53:24.871268Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:3

## Provider creates the Agreement (Provider -> Consumer)

In [107]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e",
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:f1141641-3d14-471a-a409-515a3a61860d",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-25T10:53:20.409386Z",
    "updatedAt": "2026-03-25T10:53:25.694927Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:382d4

## Consumer verifies the agreement (Consumer -> Provider)

In [108]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e",
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:f1829f9c-cf14-4786-a2c9-9275bb5d83a1",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-25T10:53:20.546655Z",
    "updatedAt": "2026-03-25T10:53:26.552647Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:0

## Provider finalizes the negotiation (Provider -> Consumer)

In [109]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:01e59546-47f9-4f9a-9fc6-9c0be313657e",
    "providerPid": "urn:provider-pid:382d4bc8-02bd-4f87-9511-07c4cc3c0d93",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:f1141641-3d14-471a-a409-515a3a61860d",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-25T10:53:20.409386Z",
    "updatedAt": "2026-03-25T10:53:27.470081Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid

## Final agreement

In [110]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:a32ee3bf-23ec-4d81-9fe4-3fdedfd7a23b",
  "negotiationAgentProcessId": "urn:negotiation-process:f1141641-3d14-471a-a409-515a3a61860d",
  "negotiationAgentMessageId": "urn:negotiation-message:9b5a8a22-6745-421e-b5dc-5fb668a9b9a8",
  "consumerParticipantId": "did:jwk:consumer",
  "providerParticipantId": "did:jwk:provider",
  "agreementContent": {
    "@id": "urn:agreement:a32ee3bf-23ec-4d81-9fe4-3fdedfd7a23b",
    "@type": "Agreement",
    "assignee": "did:jwk:consumer",
    "assigner": "did:jwk:provider",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "target": "urn:dataset:8b0488c4-d4ba-4464-9fe7-83ae68bcc2ca",
    "timestamp": "1774436005"
  },
  "target": "urn:dataset:8b0488c4-d4ba-4464-9fe7-83ae68bcc2ca",
  "state": "ACTIVE",
  "createdAt": "2026-03-25T10:53:25.701202Z",
  "updatedAt": "2026-03-25T10:53:27.476459Z"
}

Final agreement id: 
urn:agreement:a32ee3bf-23ec-4d81-9fe4-3fdedfd7a23b



# Transfer Negotiation with Agents


## Transfer request setup (Consumer -> Provider)


In [111]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+asd",
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["response"]["consumerPid"]
transfer_process_provider_pid = response_as_json["response"]["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "agreementId": "urn:agreement:a32ee3bf-23ec-4d81-9fe4-3fdedfd7a23b",
    "format": "http+asd",
    "dataAddress": null,
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531",
    "state": "REQUESTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:3accfe30-30c1-4992-bf37-17ef4d9f7115",
    "state": "REQUESTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:a32ee3bf-23ec-4d81-9fe4-3fdedfd7a23b",
    "c

## Start transfer (Provider -> Consumer)

In [112]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1100/dataplane/proxy/urn:dataplane-transfer:d83b971a-3e76-4899-ba3c-3f03fe7c7d18",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:66971d51-0667-4b65-a767-e81343c7d53b",
    "state": "STARTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "ag

## Suspend transfer (Consumer -> Provider)

In [113]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:3accfe30-30c1-4992-bf37-17ef4d9f7115",
    "state": "SUSPENDED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:a32ee3bf-23ec-4d81-9fe4-3fdedfd7a23b",
    "callbackAddress": "http://127.0.0.1:1200/dsp/cu

## Restart transfer (Consumer -> Provider)

In [114]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531",
    "state": "STARTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:3accfe30-30c1-4992-bf37-17ef4d9f7115",
    "state": "STARTED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:a32ee3bf-23ec-4d81-9fe4-3fdedfd7a23b",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt":

## Suspension by Provider (Provider -> Consumer)
(Note: Testing provider-initiated suspension)

In [115]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:66971d51-0667-4b65-a767-e81343c7d53b",
    "state": "SUSPENDED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:a32ee3bf-23ec-4d81-9fe4-3fdedfd7a23b",
    "callbackAddress": "http://127.0.0.1:1100/dsp/cu

## Failure Test: Attempt start with invalid parameters

In [116]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531"
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "5000",
    "reason": [
      "Petition Error\n"
    ]
  }
}


## Failure Test: Attempt duplicate or invalid suspension

In [117]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "5000",
    "reason": [
      "Parse Error"
    ]
  }
}


## Finalize transfer (Provider -> Consumer)

In [118]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:758454d1-1210-42ff-ba7e-18858bb3f2bf",
    "providerPid": "urn:provider-pid:c8143649-99db-47cb-a265-bfe910e8e531",
    "state": "COMPLETED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:66971d51-0667-4b65-a767-e81343c7d53b",
    "state": "COMPLETED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:a32ee3bf-23ec-4d81-9fe4-3fdedfd7a23b",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "created